# Stage 3 技術實作：基於歷史訂單之互補商品關聯規則挖掘 (FP-Growth)

## 實作目標
本技術單元旨在實作系統架構中的 **Offline 離線運算模組**。透過分析 91APP 歷史子單資料 (`OrderSlave`)，利用 **FP-Growth 演算法** 挖掘商品之間的關聯規則（Market Basket Analysis），進而產出 `relation_product`（最常被同時購買之互補商品表），作為 Stage 3 組合推薦引擎的候選集（Candidate Generation）。

## 驗證規範 (Notice)
為了確保模型評估的嚴謹度，避免時間序列上的資料洩漏（Data Leakage），本實作將**嚴格僅使用前 80% 時間區間之訂單數據**作為訓練集，保留後 20% 時間區間供後續階段進行交叉驗證（Validation）。

In [ ]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import fpgrowth, association_rules
import gc

print("--- Data Preprocessing Pipeline Activated ---")
SHOP_ID = "RZSHERLBqjPGOUFO01RYew=="
CHUNK_SIZE = 500_000
chunks = []
for chunk in pd.read_csv(f'../Order_TS_filtered_{SHOP_ID}.csv', chunksize=CHUNK_SIZE,
                         on_bad_lines='skip', low_memory=False,
                         usecols=['TradesGroupCode', 'OuterProductSkuCode', 'SalePageId', 'UnitPrice', 'OrderDateTime', 'StatusDef']):
    chunks.append(chunk[chunk['StatusDef'] == 'Finish'])

df_clean = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()

df_clean['OrderDateTime'] = pd.to_datetime(df_clean['OrderDateTime'], format='mixed')
df_clean = df_clean.dropna(subset=['OuterProductSkuCode'])
print(f"完成交易且有商品代碼筆數: {len(df_clean):,}")

# 建立 OuterProductSkuCode -> SalePageId 對照表
sku_to_salepage = (df_clean.dropna(subset=['SalePageId'])
                   .drop_duplicates('OuterProductSkuCode')
                   .set_index('OuterProductSkuCode')['SalePageId']
                   .to_dict())
print(f"有 SalePageId 對應的 SKU 數: {len(sku_to_salepage):,}")

# 建立 OuterProductSkuCode -> UnitPrice 對照表（取眾數避免促銷價干擾）
df_clean['UnitPrice'] = pd.to_numeric(df_clean['UnitPrice'], errors='coerce')
sku_to_price = (df_clean.dropna(subset=['UnitPrice'])
                .groupby('OuterProductSkuCode')['UnitPrice']
                .agg(lambda x: x.mode().iloc[0])
                .to_dict())
print(f"有 UnitPrice 對應的 SKU 數: {len(sku_to_price):,}")

df_clean = df_clean.sort_values('OrderDateTime').reset_index(drop=True)

split_idx = int(len(df_clean) * 0.8)
split_date = df_clean['OrderDateTime'].iloc[split_idx]
print(f"【時間分割點】: {split_date}")

df_train = df_clean.iloc[:split_idx].copy()
print(f"訓練集數據筆數 (前 80%): {len(df_train):,}")

del df_clean
gc.collect()

--- Data Preprocessing Pipeline Activated ---


FileNotFoundError: [Errno 2] No such file or directory: './Order_TS_filtered_RZSHERLBqjPGOUFO01RYew==.csv'

## 數據矩陣轉換 (One-Hot Encoding)
由於 FP-Growth 演算法要求輸入格式為佈林矩陣（Transaction Matrix），我們需要將同一筆主單編號 (`TradesGroupCode`) 下的所有子單商品 (`SalePageId`) 打包成購物籃，並進行 One-Hot 編碼轉換。

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from collections import Counter

print("篩選多商品訂單...")
basket_list = df_train.groupby('TradesGroupCode')['OuterProductSkuCode'].apply(list)
basket_list = basket_list[basket_list.apply(len) >= 2]
print(f"含 2 件以上商品的訂單數: {len(basket_list):,}")

# 只保留 Top 500 最高頻商品，避免稀疏矩陣過大
product_counts = Counter(p for b in basket_list for p in b)
top_products = set(p for p, _ in product_counts.most_common(500))
basket_list = basket_list.apply(lambda b: [p for p in b if p in top_products])
basket_list = basket_list[basket_list.apply(len) >= 2]
print(f"限縮至 Top 500 商品後，有效訂單數: {len(basket_list):,}")

te = TransactionEncoder()
te_array = te.fit(basket_list).transform(basket_list, sparse=True)
basket_sets = pd.DataFrame.sparse.from_spmatrix(te_array, columns=te.columns_)
print(f"稀疏矩陣建立完成。訂單數: {basket_sets.shape[0]:,}, 商品數: {basket_sets.shape[1]:,}")

gc.collect()

篩選多商品訂單...
含 2 件以上商品的訂單數: 332,984
限縮至 Top 500 商品後，有效訂單數: 329,594
稀疏矩陣建立完成。訂單數: 329,594, 商品數: 500


0

## FP-Growth 演算法執行與關聯商品表 (`relation_product`) 產出
在此步驟中，我們設定最小支持度（`min_support`）與最小信賴度（`min_threshold`），透過 `mlxtend` 跑出商品間的強關聯規則，並將其格式化為系統架構所需的互補商品關聯對照表。

In [ ]:
frequent_itemsets = fpgrowth(basket_sets, min_support=0.001, use_colnames=True)
print(f"頻繁模式樹建立成功，共挖掘出 {len(frequent_itemsets)} 組頻繁商品集。")

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0,
                          num_itemsets=len(frequent_itemsets))
print(f"關聯規則篩選完成，共生成 {len(rules)} 條有效規則。")

rules = rules[
    (rules['antecedents'].apply(len) == 1) &
    (rules['consequents'].apply(len) == 1)
].copy()

rules['antecedent_id'] = rules['antecedents'].apply(lambda x: list(x)[0])
rules['consequent_id'] = rules['consequents'].apply(lambda x: list(x)[0])

relation_product = (rules[['antecedent_id', 'consequent_id', 'support', 'confidence', 'lift']]
                    .reset_index(drop=True))
relation_product.columns = ['target_product_id', 'complementary_product_id', 'support', 'confidence', 'lift']

relation_product['target_salepage_id'] = (relation_product['target_product_id']
                                          .map(sku_to_salepage)
                                          .astype('Int64'))
relation_product['complementary_salepage_id'] = (relation_product['complementary_product_id']
                                                  .map(sku_to_salepage)
                                                  .astype('Int64'))

# 過濾掉同一個 SalePageId 的規則
before = len(relation_product)
relation_product = relation_product[
    relation_product['target_salepage_id'].isna() |
    relation_product['complementary_salepage_id'].isna() |
    (relation_product['target_salepage_id'] != relation_product['complementary_salepage_id'])
].reset_index(drop=True)
print(f"過濾同 SalePageId 規則：{before} → {len(relation_product)} 筆")

# 讀取 SalePage 商品名稱
df_salepage = pd.read_csv('./SalePage.csv', usecols=['SalePageId', 'SalePageTitle'], encoding='utf-8-sig')
salepage_title = df_salepage.set_index('SalePageId')['SalePageTitle'].to_dict()

relation_product['target_title'] = relation_product['target_salepage_id'].map(salepage_title)
relation_product['complementary_title'] = relation_product['complementary_salepage_id'].map(salepage_title)

# 過濾掉滿額贈商品
before = len(relation_product)
relation_product = relation_product[
    ~relation_product['target_title'].fillna('').str.startswith('【滿額贈】') &
    ~relation_product['complementary_title'].fillna('').str.startswith('【滿額贈】')
].reset_index(drop=True)
print(f"過濾滿額贈規則：{before} → {len(relation_product)} 筆")

# 帶入價格
relation_product['target_price'] = relation_product['target_product_id'].map(sku_to_price)
relation_product['complementary_price'] = relation_product['complementary_product_id'].map(sku_to_price)

# 過濾掉互補商品單價為 0 的規則
before = len(relation_product)
relation_product = relation_product[
    relation_product['complementary_price'].notna() &
    (relation_product['complementary_price'] > 0)
].reset_index(drop=True)
print(f"過濾互補商品單價為 0 規則：{before} → {len(relation_product)} 筆")

# 綜合分數：兼顧流行度(confidence)與真正互補性(lift)
relation_product['score'] = relation_product['confidence'] * np.log(relation_product['lift'])
relation_product = relation_product.sort_values('score', ascending=False).reset_index(drop=True)

# 欄位順序
cols = ['target_title', 'complementary_title',
        'target_price', 'complementary_price',
        'target_product_id', 'complementary_product_id',
        'target_salepage_id', 'complementary_salepage_id',
        'score', 'confidence', 'lift', 'support']
relation_product = relation_product[cols]

relation_product.to_csv(f'./relation_product_{SHOP_ID}.csv', index=False)
print(f"\n--- Success: relation_product table generated successfully! ---")
print(f"共計 {len(relation_product):,} 筆強關聯規則。")
print(f"其中有 target_salepage_id: {relation_product['target_salepage_id'].notna().sum():,} 筆")
print(f"其中有 complementary_salepage_id: {relation_product['complementary_salepage_id'].notna().sum():,} 筆")

relation_product.head(20)

頻繁模式樹建立成功，共挖掘出 1560 組頻繁商品集。
關聯規則篩選完成，共生成 3630 條有效規則。
過濾同 SalePageId 規則：1100 → 1100 筆
過濾滿額贈規則：1100 → 664 筆
過濾互補商品單價為 0 規則：664 → 600 筆

--- Success: relation_product table generated successfully! ---
共計 600 筆強關聯規則。
其中有 target_salepage_id: 600 筆
其中有 complementary_salepage_id: 600 筆


,target_title,complementary_title,target_price,complementary_price,target_product_id,complementary_product_id,target_salepage_id,complementary_salepage_id,score,confidence,lift,support
0,【超值加價購】100%純棉舒敏潔膚巾(十字紋親膚版100pcs),【超值加價購】100%純棉植物纖維潔膚巾 (珍珠紋加厚版100pcs),99.0,99.0,MpPR6SSjumTYypqYQ79weA==,8X6604XYgyhDZc/OlaA8FA==,9029388,7534173,0.826080,0.507359,5.094675,0.005543
1,一夜透亮活膚精露,頂級藍銅肌因修復精華【最高濃度5000ppm】,2480.0,2980.0,PFTPZPvNdRmHQ9dv3Vm/5Q==,s6lH2IdGEWyFEea99W3f9Q==,8518508,8558584,0.752169,0.254554,19.198944,0.001696
2,一夜透亮活膚精露,熱療撫紋導入儀（音波震動版）,2480.0,1680.0,PFTPZPvNdRmHQ9dv3Vm/5Q==,E8k4fwisTFo8B2R41VMuLA==,8518508,8522342,0.743840,0.218124,30.270449,0.001453
3,熱療撫紋導入儀（音波震動版）,一夜透亮活膚精露,1680.0,2480.0,E8k4fwisTFo8B2R41VMuLA==,PFTPZPvNdRmHQ9dv3Vm/5Q==,8522342,8518508,0.687778,0.201684,30.270449,0.001453
4,舒緩安敏原生露,76酵母水楊酸身體乳【蔡頤榛推薦】,1380.0,1280.0,lpbbhxSdxr7PasMBNVWmFg==,ZX8EX1Xe82ViDM61FEcreg==,9029426,8943281,0.636376,0.257189,11.873942,0.001438
5,【爆水小藍瓶】藍銅玻尿酸保濕修復精華【謝京穎推薦】,76酵母胺基酸淨膚潔顏露（防滑升級版）【鍾明軒推薦】,1980.0,1080.0,7tn6GK4fmn6WKWx6kL6U5A==,U1s83W/THfiF2QabuQmVQw==,8192135,7467069,0.563891,0.426634,3.749863,0.014415
6,【混和肌救星】99控油保濕平衡乳【黃上晏推薦】,76酵母胺基酸淨膚潔顏露（防滑升級版）【鍾明軒推薦】,1280.0,1080.0,Gz28CMNQ2Xpg8e/KZZfXxQ==,U1s83W/THfiF2QabuQmVQw==,7904452,7467069,0.557712,0.423969,3.726439,0.034819
7,【任2件$2022】蜂王乳極潤修護精華露【國際名模推薦】,黃金胜肽賦活精華【鄭亦真推薦】,1280.0,1680.0,aSjLmAHGeSBQidn7SYFtZA==,IVnoXdWXrQoahlOQdtP9ow==,7534436,7183547,0.516125,0.325504,4.882316,0.015935
8,高效全能水感防曬乳【SPF50+PA++++】,76酵母胺基酸淨膚潔顏露（防滑升級版）【鍾明軒推薦】,1280.0,1080.0,JCkUZa54DO/iNZHQRlZJ5Q==,U1s83W/THfiF2QabuQmVQw==,8776959,7467069,0.485852,0.392414,3.449089,0.009794
9,黃金修護白泡泡面膜【毛孔吸塵器．11點熱吵店推薦】,76酵母胺基酸淨膚潔顏露（防滑升級版）【鍾明軒推薦】,1280.0,1080.0,6Q5oMaDdU8oUtCvzwmajjg==,U1s83W/THfiF2QabuQmVQw==,7353241,7467069,0.475213,0.387647,3.407192,0.026317


In [ ]:
same = relation_product[
    relation_product['target_salepage_id'].notna() &
    relation_product['complementary_salepage_id'].notna() &
    (relation_product['target_salepage_id'] == relation_product['complementary_salepage_id'])
]
print(f"兩個 SalePageId 相同的規則數: {len(same)} 筆")
same

兩個 SalePageId 相同的規則數: 0 筆


,target_title,complementary_title,target_price,complementary_price,target_product_id,complementary_product_id,target_salepage_id,complementary_salepage_id,score,confidence,lift,support


In [ ]:
target_products = set(relation_product['target_product_id'].unique())
comp_products = set(relation_product['complementary_product_id'].unique())

print(f"target 側不同商品數:        {len(target_products):,}")
print(f"complementary 側不同商品數: {len(comp_products):,}")
print(f"兩側聯集（總不同商品數）:    {len(target_products | comp_products):,}")
print(f"兩側都出現的商品數:          {len(target_products & comp_products):,}")

target 側不同商品數:        84
complementary 側不同商品數: 84
兩側聯集（總不同商品數）:    84
兩側都出現的商品數:          84
